# 19 - Optical Flow Explanation Faithfulness

This notebook loads the compressed Grad-CAM outputs from notebook 05 and evaluates whether saliency follows left-ventricular motion. It does **not** regenerate Grad-CAM and does **not** rerun model inference. Optical flow is computed once per unique saved frame sequence and reused across all matching models/layers.

In [ ]:
# Kaggle setup. Skip if your image already has these packages.
%pip install -q opencv-python-headless pandas matplotlib scipy tqdm torchvision

In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import math
import os
import random
import sys
from typing import Any

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import torchvision
    from torchvision.models.optical_flow import Raft_Large_Weights, raft_large
except Exception as exc:
    torchvision = None
    Raft_Large_Weights = None
    raft_large = None
    print(f"TorchVision RAFT unavailable; set CONFIG['flow_method']='opencv_farneback' if needed. Error: {exc}")

from src.dataset import split_by_echonet_filelist
from src.temporal_dataset_variable_stride import load_temporal_metadata, build_fps_lookup
from src.utils import load_echonet_tables, set_seed

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Configuration

`GRADCAM_RUN_DIR` should point to the output folder from notebook 05. This folder must contain `tables/cam_manifest.csv` and `cams/`. The notebook will use the same visualization samples recorded by notebook 05 when possible.

In [ ]:
RUN_MODE = "smoke"  # "smoke" or "full"
SMOKE_SAMPLE_LIMIT = 3
MAX_QUALITATIVE_SAMPLES = 10
RANDOM_SEED = 42

RAW_DIR = Path(os.environ.get("ECHONET_RAW_DIR", PROJECT_ROOT / "data" / "raw" / "EchoNet-Dynamic"))
PROCESSED_DIR = Path(os.environ.get("ECHONET_PROCESSED_DIR", PROJECT_ROOT / "data" / "processed"))
VIDEOS_DIR = RAW_DIR / "Videos"

GRADCAM_RUN_DIR = Path(os.environ.get(
    "GRADCAM_RUN_DIR",
    "/kaggle/input/gradcam-temporal-evaluation/gradcam_temporal_evaluation"
    if Path("/kaggle/input").exists()
    else PROJECT_ROOT / "outputs" / "runs" / "gradcam_temporal_evaluation",
))
PSEUDOLABEL_RUN_DIR = Path(os.environ.get(
    "PSEUDOLABEL_RUN_DIR",
    "/kaggle/input/segmentation-pseudolabels/segmentation_pseudolabels"
    if Path("/kaggle/input").exists()
    else PROJECT_ROOT / "outputs" / "runs" / "segmentation_pseudolabels",
))

OUTPUT_DIR = Path("/kaggle/working/outputs/runs/optical_flow_explanation_faithfulness") if Path("/kaggle/working").exists() else PROJECT_ROOT / "outputs" / "runs" / "optical_flow_explanation_faithfulness"
METRICS_DIR = OUTPUT_DIR / "metrics"
FLOW_DIR = OUTPUT_DIR / "flow"
QUAL_DIR = OUTPUT_DIR / "qualitative_examples"
MANIFEST_DIR = OUTPUT_DIR / "manifests"
for directory in [METRICS_DIR, FLOW_DIR, QUAL_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    # Reuses notebook 16's RAFT-style flow by default. Use "opencv_farneback" if RAFT is unavailable/too slow.
    "flow_method": "raft_large",  # "raft_large" or "opencv_farneback"
    "raft_transition_batch_size": 8,
    "flow_storage_dtype": "float16",
    "image_size": [112, 112],
    "analysis_cam_key": "positive_clip_normalized_uint8",
    "topk_fraction": 0.10,
    "moving_region_percentile": 75.0,
    "valid_flow_error_threshold_px": 3.0,
    "visualization_transition_count": 4,
    "prefer_layers": [
        "convlstm_bottleneck_encoder",
        "convlstm_temporal_bottleneck",
        "convlstm_decoder3_sliding_window",
        "unet_encoder_bottleneck",
        "unet_final_convolution",
    ],
}

with (OUTPUT_DIR / "config.json").open("w", encoding="utf-8") as f:
    json.dump({**CONFIG, "gradcam_run_dir": str(GRADCAM_RUN_DIR), "pseudolabel_run_dir": str(PSEUDOLABEL_RUN_DIR)}, f, indent=2)

print(f"Grad-CAM run directory: {GRADCAM_RUN_DIR}")
print(f"Pseudolabel run directory: {PSEUDOLABEL_RUN_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## Load Grad-CAM Manifests

Notebook 05 saves one row per sample/model/layer in `tables/cam_manifest.csv`. This notebook remaps stale absolute paths by filename when notebook 05 outputs have been uploaded as a Kaggle dataset.

In [ ]:
cam_manifest_path = GRADCAM_RUN_DIR / "tables" / "cam_manifest.csv"
assert cam_manifest_path.exists(), f"Missing notebook 05 CAM manifest: {cam_manifest_path}"
cam_manifest = pd.read_csv(cam_manifest_path)
assert not cam_manifest.empty, "CAM manifest is empty. Run notebook 05 first."

# Build a filename index so stale /kaggle/working paths in the manifest can be remapped after upload.
cam_file_index = {path.name: path for path in (GRADCAM_RUN_DIR / "cams").rglob("*.npz")}

def resolve_cam_path(path_text: str) -> Path:
    raw = Path(str(path_text))
    if raw.exists():
        return raw
    if raw.name in cam_file_index:
        return cam_file_index[raw.name]
    candidate = GRADCAM_RUN_DIR / "cams" / raw.name
    if candidate.exists():
        return candidate
    raise FileNotFoundError(f"Could not resolve CAM NPZ path: {path_text}")

cam_manifest["resolved_cam_npz_path"] = [str(resolve_cam_path(p)) for p in cam_manifest["cam_npz_path"]]

# Keep the primary ConvLSTM layers first, but retain all layers if available.
if "target_layer" not in cam_manifest.columns:
    raise ValueError("cam_manifest.csv must contain target_layer.")

if RUN_MODE == "smoke":
    rng = random.Random(RANDOM_SEED)
    sample_ids = sorted(cam_manifest["sample_id"].astype(str).unique())
    keep_samples = set(rng.sample(sample_ids, min(SMOKE_SAMPLE_LIMIT, len(sample_ids))))
    cam_manifest = cam_manifest[cam_manifest["sample_id"].astype(str).isin(keep_samples)].reset_index(drop=True)

print(f"CAM rows selected: {len(cam_manifest):,}")
display(cam_manifest.head())

## Select Same Representative Samples As Notebook 05

If notebook 05 wrote `qualitative_visualization_manifest.csv`, those sample IDs are reused. Otherwise the notebook selects up to 10 samples from the CAM manifest.

In [ ]:
qual_manifest_path = GRADCAM_RUN_DIR / "tables" / "qualitative_visualization_manifest.csv"
if qual_manifest_path.exists():
    q = pd.read_csv(qual_manifest_path)
    visualization_sample_ids = list(dict.fromkeys(q["sample_id"].astype(str).tolist()))[:MAX_QUALITATIVE_SAMPLES]
else:
    visualization_sample_ids = list(dict.fromkeys(cam_manifest["sample_id"].astype(str).tolist()))[:MAX_QUALITATIVE_SAMPLES]

if RUN_MODE == "smoke":
    visualization_sample_ids = visualization_sample_ids[: min(len(visualization_sample_ids), SMOKE_SAMPLE_LIMIT)]

print(f"Visualization samples: {visualization_sample_ids}")
pd.DataFrame({"sample_id": visualization_sample_ids}).to_csv(MANIFEST_DIR / "selected_visualization_samples.csv", index=False)

## Video Frames And Pseudo-Label Loader

Pseudo-labels are optional but recommended for LV-restricted motion metrics. If unavailable, LV-specific metrics are recorded as `NaN` and whole-frame metrics still run.

In [ ]:
def parse_frame_indices(value: Any) -> list[int]:
    if isinstance(value, str):
        return [int(x) for x in value.split() if str(x).strip()]
    if isinstance(value, (list, tuple, np.ndarray)):
        return [int(x) for x in value]
    return []


def video_path(video_id: str) -> Path:
    filename = video_id if str(video_id).lower().endswith(".avi") else f"{video_id}.avi"
    return VIDEOS_DIR / filename


def read_video_frames(video_id: str, frame_indices: list[int], image_size: tuple[int, int]) -> np.ndarray:
    path = video_path(video_id)
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {path}")
    frames = []
    H, W = image_size
    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame_bgr = cap.read()
        if not ok or frame_bgr is None:
            cap.release()
            raise ValueError(f"Could not read frame {frame_idx} from {path}")
        gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
        gray = cv2.resize(gray, (W, H), interpolation=cv2.INTER_AREA)
        frames.append(gray.astype(np.float32) / 255.0)
    cap.release()
    return np.stack(frames, axis=0).astype(np.float32)


class DensePseudolabelStore:
    def __init__(self, run_dir: Path):
        self.available = False
        self.run_dir = Path(run_dir)
        manifest_path = self.run_dir / "manifests" / "pseudolabel_frame_manifest.csv"
        if not manifest_path.exists():
            print(f"Pseudo-label manifest not found: {manifest_path}. LV-restricted metrics will be NaN.")
            self.frame_manifest = pd.DataFrame()
            self.lookup = {}
            self._cache = {}
            return
        self.available = True
        self.frame_manifest = pd.read_csv(manifest_path)
        self.lookup = {}
        for row in self.frame_manifest.itertuples(index=False):
            raw_path = Path(str(row.mask_npz_path))
            if raw_path.exists():
                path = raw_path
            else:
                remapped = self.run_dir / "masks_by_video" / raw_path.name
                path = remapped if remapped.exists() else raw_path
            self.lookup[(str(row.video_id), int(row.frame_idx))] = (path, int(row.mask_row_index), bool(row.is_ground_truth))
        self._cache = {}

    def _load_npz(self, path: Path) -> dict[str, np.ndarray]:
        if path not in self._cache:
            with np.load(path, allow_pickle=False) as data:
                shape = tuple(data["mask_shape"].astype(int).tolist())
                packed = data["masks_packed"]
                masks = np.unpackbits(packed, axis=-1)[..., : shape[-1]].reshape(shape).astype(bool)
                self._cache[path] = {"masks": masks}
        return self._cache[path]

    def get(self, video_id: str, frame_idx: int) -> tuple[np.ndarray | None, bool]:
        if not self.available:
            return None, False
        key = (str(video_id), int(frame_idx))
        if key not in self.lookup:
            return None, False
        path, row_idx, is_gt = self.lookup[key]
        payload = self._load_npz(path)
        return payload["masks"][row_idx].astype(bool), bool(is_gt)

    def stack_for_frames(self, video_id: str, frame_indices: list[int]) -> tuple[np.ndarray | None, list[bool]]:
        masks = []
        sources = []
        for idx in frame_indices:
            mask, is_gt = self.get(video_id, idx)
            if mask is None:
                return None, []
            masks.append(mask)
            sources.append(is_gt)
        return np.stack(masks, axis=0).astype(bool), sources


pseudolabel_store = DensePseudolabelStore(PSEUDOLABEL_RUN_DIR)
print(f"Pseudo-labels available: {pseudolabel_store.available}")

## Optical Flow Implementation

This reuses notebook 16's RAFT-style preprocessing: grayscale `[T,H,W]` frames are expanded to RGB, resized/padded to satisfy RAFT's minimum 128x128 correlation-pyramid requirement, then flows are cropped/resized back to the ConvLSTM/U-Net input resolution. Set `CONFIG['flow_method']='opencv_farneback'` for a faster CPU/GPU-light fallback.

In [ ]:
raft_model = None
raft_weights = None
if CONFIG["flow_method"] == "raft_large":
    assert raft_large is not None and Raft_Large_Weights is not None, "TorchVision RAFT is unavailable. Set flow_method='opencv_farneback'."
    raft_weights = Raft_Large_Weights.DEFAULT
    raft_model = raft_large(weights=raft_weights, progress=True).to(device).eval()
    print(f"Loaded RAFT-Large weights: {raft_weights}")


def pad_to_raft_size(tensor: torch.Tensor, min_size: int = 128, multiple: int = 8) -> tuple[torch.Tensor, dict[str, int]]:
    _, _, h, w = tensor.shape
    target_h = max(min_size, int(math.ceil(h / multiple) * multiple))
    target_w = max(min_size, int(math.ceil(w / multiple) * multiple))
    pad_h = target_h - h
    pad_w = target_w - w
    padded = F.pad(tensor, (0, pad_w, 0, pad_h), mode="replicate")
    return padded, {"orig_h": h, "orig_w": w, "target_h": target_h, "target_w": target_w, "pad_h": pad_h, "pad_w": pad_w}


def crop_flow_to_original(flow: torch.Tensor, pad_info: dict[str, int]) -> torch.Tensor:
    flow = flow[..., : pad_info["orig_h"], : pad_info["orig_w"]]
    return flow


def frames_to_raft_tensors(frames: np.ndarray) -> tuple[torch.Tensor, torch.Tensor, dict[str, int]]:
    # [T,H,W] grayscale [0,1] -> transition pairs [T-1,3,H,W]
    rgb = np.repeat(frames[:, None], 3, axis=1).astype(np.float32)
    img1 = torch.from_numpy(rgb[:-1]).to(device)
    img2 = torch.from_numpy(rgb[1:]).to(device)
    img1, pad_info = pad_to_raft_size(img1)
    img2, _ = pad_to_raft_size(img2)
    preprocess = raft_weights.transforms() if raft_weights is not None else None
    if preprocess is not None:
        img1, img2 = preprocess(img1, img2)
    return img1, img2, pad_info


@torch.no_grad()
def compute_raft_flows(frames: np.ndarray, batch_size: int = 8) -> tuple[np.ndarray, np.ndarray, dict[str, int]]:
    img1, img2, pad_info = frames_to_raft_tensors(frames)
    forward_chunks = []
    backward_chunks = []
    for start in range(0, img1.shape[0], batch_size):
        end = min(start + batch_size, img1.shape[0])
        a = img1[start:end]
        b = img2[start:end]
        fwd = raft_model(a, b)[-1]
        bwd = raft_model(b, a)[-1]
        forward_chunks.append(crop_flow_to_original(fwd, pad_info).detach().cpu())
        backward_chunks.append(crop_flow_to_original(bwd, pad_info).detach().cpu())
    forward = torch.cat(forward_chunks, dim=0).numpy().astype(np.float32)
    backward = torch.cat(backward_chunks, dim=0).numpy().astype(np.float32)
    return forward, backward, pad_info


def compute_farneback_flows(frames: np.ndarray) -> tuple[np.ndarray, np.ndarray, dict[str, int]]:
    frames_u8 = np.round(np.clip(frames, 0, 1) * 255).astype(np.uint8)
    forward, backward = [], []
    for t in range(len(frames_u8) - 1):
        fwd = cv2.calcOpticalFlowFarneback(frames_u8[t], frames_u8[t + 1], None, 0.5, 3, 15, 3, 5, 1.2, 0)
        bwd = cv2.calcOpticalFlowFarneback(frames_u8[t + 1], frames_u8[t], None, 0.5, 3, 15, 3, 5, 1.2, 0)
        forward.append(np.moveaxis(fwd, -1, 0))
        backward.append(np.moveaxis(bwd, -1, 0))
    return np.stack(forward).astype(np.float32), np.stack(backward).astype(np.float32), {"method": "opencv_farneback"}


def warp_uv(flow_to_warp: torch.Tensor, reference_flow: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    # flow tensors [N,2,H,W], reference maps first image -> second image
    n, _, h, w = reference_flow.shape
    yy, xx = torch.meshgrid(torch.arange(h, device=reference_flow.device), torch.arange(w, device=reference_flow.device), indexing="ij")
    x = xx[None].float() + reference_flow[:, 0]
    y = yy[None].float() + reference_flow[:, 1]
    valid = (x >= 0) & (x <= w - 1) & (y >= 0) & (y <= h - 1)
    x_norm = 2.0 * x / max(w - 1, 1) - 1.0
    y_norm = 2.0 * y / max(h - 1, 1) - 1.0
    grid = torch.stack([x_norm, y_norm], dim=-1)
    warped = F.grid_sample(flow_to_warp, grid, mode="bilinear", padding_mode="zeros", align_corners=True)
    return warped, valid


def maybe_cast(array: np.ndarray) -> np.ndarray:
    return array.astype(np.float16) if str(CONFIG["flow_storage_dtype"]).lower() == "float16" else array.astype(np.float32)


def sequence_key(video_id: str, frame_indices: list[int]) -> str:
    text = str(video_id) + "_" + "_".join(str(x) for x in frame_indices)
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:16]

## Build Unique Sequence Table

Optical flow is computed once for each unique `(video_id, frame_indices)` sequence, then reused for every model/layer whose CAMs correspond to that sequence.

In [ ]:
sequence_rows = []
for row in cam_manifest.itertuples(index=False):
    frame_indices = parse_frame_indices(row.frame_indices)
    if len(frame_indices) < 2:
        continue
    key = sequence_key(str(row.video_id), frame_indices)
    sequence_rows.append({
        "sequence_key": key,
        "sample_id": str(row.sample_id),
        "video_id": str(row.video_id),
        "frame_indices": " ".join(str(x) for x in frame_indices),
        "transition_count": len(frame_indices) - 1,
    })
sequence_df = pd.DataFrame(sequence_rows).drop_duplicates(subset=["sequence_key"]).reset_index(drop=True)
sequence_df.to_csv(MANIFEST_DIR / "unique_flow_sequences.csv", index=False)
print(f"Unique flow sequences: {len(sequence_df):,}")
display(sequence_df.head())

## Compute Flow Once Per Sequence

Full flow arrays are saved once in `flow/`. Per-transition flow summaries are saved separately in CSV.

In [ ]:
def transition_rois_from_masks(masks: np.ndarray | None) -> tuple[np.ndarray | None, np.ndarray | None]:
    if masks is None:
        return None, None
    cavity = []
    boundary = []
    kernel = np.ones((3, 3), np.uint8)
    masks_u8 = masks.astype(np.uint8)
    for t in range(masks_u8.shape[0] - 1):
        m0, m1 = masks_u8[t], masks_u8[t + 1]
        union = (m0 > 0) | (m1 > 0)
        b0 = cv2.morphologyEx(m0, cv2.MORPH_GRADIENT, kernel) > 0
        b1 = cv2.morphologyEx(m1, cv2.MORPH_GRADIENT, kernel) > 0
        boundary.append(b0 | b1)
        cavity.append(union)
    return np.stack(cavity).astype(bool), np.stack(boundary).astype(bool)


def compute_flow_for_sequence(row: Any) -> dict[str, Any]:
    frame_indices = parse_frame_indices(row.frame_indices)
    video_id = str(row.video_id)
    key = str(row.sequence_key)
    out_path = FLOW_DIR / f"{key}_flow.npz"
    analysis_path = FLOW_DIR / f"{key}_flow_analysis.npz"

    if out_path.exists() and analysis_path.exists():
        with np.load(analysis_path, allow_pickle=False) as data:
            return json.loads(str(data["summary_json"]))

    frames = read_video_frames(video_id, frame_indices, tuple(CONFIG["image_size"]))
    if CONFIG["flow_method"] == "raft_large":
        forward, backward, pad_info = compute_raft_flows(frames, int(CONFIG["raft_transition_batch_size"]))
    else:
        forward, backward, pad_info = compute_farneback_flows(frames)

    fwd_t = torch.from_numpy(forward).to(device)
    bwd_t = torch.from_numpy(backward).to(device)
    warped_bwd, valid = warp_uv(bwd_t, fwd_t)
    fb_error = torch.linalg.norm(fwd_t + warped_bwd, dim=1).detach().cpu().numpy().astype(np.float32)
    valid_mask = (valid.detach().cpu().numpy() & (fb_error <= float(CONFIG["valid_flow_error_threshold_px"]))).astype(bool)

    mag = np.linalg.norm(forward, axis=1).astype(np.float32)
    angle = np.arctan2(forward[:, 1], forward[:, 0]).astype(np.float32)
    masks, mask_is_gt = pseudolabel_store.stack_for_frames(video_id, frame_indices)
    cavity_roi, boundary_roi = transition_rois_from_masks(masks)
    if cavity_roi is None:
        cavity_roi_to_save = np.zeros_like(mag, dtype=np.uint8)
        boundary_roi_to_save = np.zeros_like(mag, dtype=np.uint8)
    else:
        cavity_roi_to_save = cavity_roi.astype(np.uint8)
        boundary_roi_to_save = boundary_roi.astype(np.uint8)

    np.savez_compressed(
        out_path,
        frames=maybe_cast(frames),
        flow_forward_uv=maybe_cast(forward),
        flow_backward_uv=maybe_cast(backward),
        flow_forward_magnitude=maybe_cast(mag),
        flow_forward_angle=maybe_cast(angle),
        forward_backward_error=maybe_cast(fb_error),
        valid_flow_mask=valid_mask.astype(np.uint8),
        transition_cavity_roi=cavity_roi_to_save,
        transition_boundary_roi=boundary_roi_to_save,
        frame_indices=np.asarray(frame_indices, dtype=np.int32),
        metadata_json=np.asarray(json.dumps({"video_id": video_id, "sequence_key": key, "flow_method": CONFIG["flow_method"], "pad_info": pad_info}, default=str)),
    )

    transition_rows = []
    for t in range(mag.shape[0]):
        roi = cavity_roi[t] if cavity_roi is not None else np.ones_like(mag[t], dtype=bool)
        valid_roi = valid_mask[t] & roi
        transition_rows.append({
            "sequence_key": key,
            "sample_id": str(row.sample_id),
            "video_id": video_id,
            "transition_idx": int(t),
            "start_frame_idx": int(frame_indices[t]),
            "end_frame_idx": int(frame_indices[t + 1]),
            "mean_flow_magnitude": float(np.nanmean(mag[t])),
            "mean_lv_flow_magnitude": float(np.nanmean(mag[t][valid_roi])) if valid_roi.any() else float("nan"),
            "valid_flow_fraction": float(valid_mask[t].mean()),
            "lv_roi_available": bool(cavity_roi is not None),
        })

    summary = {
        "sequence_key": key,
        "sample_id": str(row.sample_id),
        "video_id": video_id,
        "frame_indices": " ".join(str(x) for x in frame_indices),
        "flow_npz_path": str(out_path),
        "flow_analysis_npz_path": str(analysis_path),
        "transition_count": int(mag.shape[0]),
        "mean_flow_magnitude": float(np.nanmean(mag)),
        "valid_flow_fraction": float(valid_mask.mean()),
        "lv_roi_available": bool(cavity_roi is not None),
        "_transition_rows": transition_rows,
    }
    np.savez_compressed(analysis_path, summary_json=np.asarray(json.dumps({k: v for k, v in summary.items() if k != "_transition_rows"}, default=str)))
    return summary

flow_rows = []
flow_transition_rows = []
for row in tqdm(sequence_df.itertuples(index=False), total=len(sequence_df), desc="compute optical flow"):
    payload = compute_flow_for_sequence(row)
    flow_transition_rows.extend(payload.pop("_transition_rows", []))
    flow_rows.append(payload)

flow_df = pd.DataFrame(flow_rows)
flow_transition_df = pd.DataFrame(flow_transition_rows)
flow_df.to_csv(MANIFEST_DIR / "flow_manifest.csv", index=False)
flow_transition_df.to_csv(METRICS_DIR / "flow_transition_metrics.csv", index=False)
display(flow_df.head())

## Grad-CAM / Flow Metrics

Transition saliency is aligned to optical-flow transitions using `S_t = (CAM_t + CAM_{t+1}) / 2`. Quantitative metrics use notebook 05's positive clip-normalized CAMs.

In [ ]:
def load_cam_array(path: Path, key: str) -> np.ndarray:
    with np.load(path, allow_pickle=False) as data:
        arr = data[key]
    if arr.dtype == np.uint8:
        arr = arr.astype(np.float32) / 255.0
    else:
        arr = arr.astype(np.float32)
    return arr


def transition_saliency(cams: np.ndarray) -> np.ndarray:
    return ((cams[:-1] + cams[1:]) * 0.5).astype(np.float32)


def corr_values(a: np.ndarray, b: np.ndarray) -> tuple[float, float]:
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    if len(a) < 3 or np.nanstd(a) <= 0 or np.nanstd(b) <= 0:
        return float("nan"), float("nan")
    return float(pearsonr(a, b).statistic), float(spearmanr(a, b).statistic)


def topk_mask(values: np.ndarray, fraction: float) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0 or float(np.nanmax(values)) <= 0:
        return np.zeros_like(values, dtype=bool)
    k = max(1, int(round(values.size * float(fraction))))
    thresh = np.partition(values.ravel(), -k)[-k]
    return values >= thresh


def binary_overlap_metrics(a: np.ndarray, b: np.ndarray) -> tuple[float, float]:
    a = a.astype(bool)
    b = b.astype(bool)
    inter = float(np.logical_and(a, b).sum())
    union = float(np.logical_or(a, b).sum())
    dice = 2.0 * inter / (float(a.sum()) + float(b.sum()) + 1e-8)
    iou = inter / (union + 1e-8)
    return dice, iou


def warp_scalar_map(map_t: np.ndarray, flow_uv: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # Warp CAM_t forward using flow t->t+1. map_t [H,W], flow_uv [2,H,W]
    h, w = map_t.shape
    map_tensor = torch.from_numpy(map_t[None, None].astype(np.float32)).to(device)
    flow_tensor = torch.from_numpy(flow_uv[None].astype(np.float32)).to(device)
    yy, xx = torch.meshgrid(torch.arange(h, device=device), torch.arange(w, device=device), indexing="ij")
    x = xx[None].float() + flow_tensor[:, 0]
    y = yy[None].float() + flow_tensor[:, 1]
    valid = (x >= 0) & (x <= w - 1) & (y >= 0) & (y <= h - 1)
    grid = torch.stack([2.0 * x / max(w - 1, 1) - 1.0, 2.0 * y / max(h - 1, 1) - 1.0], dim=-1)
    warped = F.grid_sample(map_tensor, grid, mode="bilinear", padding_mode="zeros", align_corners=True)
    return warped[0, 0].detach().cpu().numpy(), valid[0].detach().cpu().numpy()


def metrics_for_cam_row(row: Any, flow_lookup: dict[str, Any]) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    frame_indices = parse_frame_indices(row.frame_indices)
    key = sequence_key(str(row.video_id), frame_indices)
    flow_info = flow_lookup[key]
    flow_path = Path(flow_info["flow_npz_path"])
    cam_path = Path(row.resolved_cam_npz_path)
    cams = load_cam_array(cam_path, str(CONFIG["analysis_cam_key"]))
    sal = transition_saliency(cams)
    with np.load(flow_path, allow_pickle=False) as flow:
        mag = flow["flow_forward_magnitude"].astype(np.float32)
        uv = flow["flow_forward_uv"].astype(np.float32)
        valid = flow["valid_flow_mask"].astype(bool)
        cavity = flow["transition_cavity_roi"].astype(bool)
        boundary = flow["transition_boundary_roi"].astype(bool)

    assert sal.shape == mag.shape, (sal.shape, mag.shape, cam_path, flow_path)
    spatial_rows = []
    temporal_rows = []
    layer = str(row.target_layer)
    model_id = str(row.model_id)
    sample_id = str(row.sample_id)
    model_family = str(row.model_family)

    for t in range(sal.shape[0]):
        base_regions = {
            "whole_frame": valid[t],
            "lv_cavity": valid[t] & cavity[t],
            "lv_boundary": valid[t] & boundary[t],
        }
        moving_thresh = np.nanpercentile(mag[t][valid[t]], float(CONFIG["moving_region_percentile"])) if valid[t].any() else np.nan
        moving_region = valid[t] & (mag[t] >= moving_thresh) if np.isfinite(moving_thresh) else np.zeros_like(valid[t], dtype=bool)
        lv_moving_region = moving_region & cavity[t]

        for region_name, region_mask in base_regions.items():
            if not region_mask.any():
                pear, spear = float("nan"), float("nan")
                weighted_flow = float("nan")
                motion_weighted_sal = float("nan")
                top_dice = float("nan")
                top_iou = float("nan")
            else:
                sal_region = sal[t][region_mask]
                mag_region = mag[t][region_mask]
                pear, spear = corr_values(sal_region, mag_region)
                weighted_flow = float(np.sum(sal_region * mag_region) / max(float(np.sum(sal_region)), 1e-8))
                motion_weighted_sal = float(np.sum(sal_region * mag_region) / max(float(np.sum(mag_region)), 1e-8))
                top_dice, top_iou = binary_overlap_metrics(
                    topk_mask(sal_region, CONFIG["topk_fraction"]),
                    topk_mask(mag_region, CONFIG["topk_fraction"]),
                )
            spatial_rows.append({
                "sample_id": sample_id,
                "sequence_key": key,
                "model_family": model_family,
                "model_id": model_id,
                "target_layer": layer,
                "transition_idx": int(t),
                "start_frame_idx": int(frame_indices[t]),
                "end_frame_idx": int(frame_indices[t + 1]),
                "region": region_name,
                "saliency_weighted_flow_magnitude": weighted_flow,
                "motion_weighted_saliency": motion_weighted_sal,
                "pearson_flow_saliency": pear,
                "spearman_flow_saliency": spear,
                "topk_dice": top_dice,
                "topk_iou": top_iou,
                "mean_saliency": float(np.nanmean(sal[t][region_mask])) if region_mask.any() else float("nan"),
                "mean_flow_magnitude": float(np.nanmean(mag[t][region_mask])) if region_mask.any() else float("nan"),
                "region_area_fraction": float(region_mask.mean()),
            })

        total_sal = float(np.sum(sal[t]))
        sal_in_moving = float(np.sum(sal[t][moving_region]) / max(total_sal, 1e-8))
        sal_in_lv_moving = float(np.sum(sal[t][lv_moving_region]) / max(total_sal, 1e-8)) if cavity[t].any() else float("nan")
        warped, warp_valid = warp_scalar_map(cams[t], uv[t])
        warp_mask = warp_valid & np.isfinite(warped) & np.isfinite(cams[t + 1])
        warp_pear, warp_spear = corr_values(warped[warp_mask], cams[t + 1][warp_mask])
        temporal_rows.append({
            "sample_id": sample_id,
            "sequence_key": key,
            "model_family": model_family,
            "model_id": model_id,
            "target_layer": layer,
            "transition_idx": int(t),
            "start_frame_idx": int(frame_indices[t]),
            "end_frame_idx": int(frame_indices[t + 1]),
            "mean_transition_saliency": float(np.nanmean(sal[t])),
            "mean_flow_magnitude": float(np.nanmean(mag[t][valid[t]])) if valid[t].any() else float("nan"),
            "saliency_fraction_in_high_motion": sal_in_moving,
            "saliency_fraction_in_high_motion_lv": sal_in_lv_moving,
            "flow_warped_cam_pearson": warp_pear,
            "flow_warped_cam_spearman": warp_spear,
        })
    return spatial_rows, temporal_rows

flow_lookup = {str(row.sequence_key): row._asdict() for row in flow_df.itertuples(index=False)}
spatial_metric_rows = []
temporal_metric_rows = []
for row in tqdm(cam_manifest.itertuples(index=False), total=len(cam_manifest), desc="compare CAMs with flow"):
    spatial, temporal = metrics_for_cam_row(row, flow_lookup)
    spatial_metric_rows.extend(spatial)
    temporal_metric_rows.extend(temporal)

spatial_metrics_df = pd.DataFrame(spatial_metric_rows)
temporal_metrics_df = pd.DataFrame(temporal_metric_rows)
spatial_metrics_df.to_csv(METRICS_DIR / "motion_saliency_spatial_transition_metrics.csv", index=False)
temporal_metrics_df.to_csv(METRICS_DIR / "motion_saliency_temporal_transition_metrics.csv", index=False)
display(spatial_metrics_df.head())

## Per-Sample And Dataset Summaries

The primary output remains per-transition/per-model/per-layer rows. Summaries are secondary convenience tables.

In [ ]:
spatial_sample_summary = (
    spatial_metrics_df
    .groupby(["sample_id", "model_family", "model_id", "target_layer", "region"], dropna=False)
    .agg(
        saliency_weighted_flow_magnitude_mean=("saliency_weighted_flow_magnitude", "mean"),
        motion_weighted_saliency_mean=("motion_weighted_saliency", "mean"),
        pearson_flow_saliency_mean=("pearson_flow_saliency", "mean"),
        spearman_flow_saliency_mean=("spearman_flow_saliency", "mean"),
        topk_dice_mean=("topk_dice", "mean"),
        topk_iou_mean=("topk_iou", "mean"),
        transition_count=("transition_idx", "count"),
    )
    .reset_index()
)

temporal_sample_summary = (
    temporal_metrics_df
    .groupby(["sample_id", "model_family", "model_id", "target_layer"], dropna=False)
    .agg(
        saliency_fraction_in_high_motion_mean=("saliency_fraction_in_high_motion", "mean"),
        saliency_fraction_in_high_motion_lv_mean=("saliency_fraction_in_high_motion_lv", "mean"),
        flow_warped_cam_pearson_mean=("flow_warped_cam_pearson", "mean"),
        flow_warped_cam_spearman_mean=("flow_warped_cam_spearman", "mean"),
        transition_count=("transition_idx", "count"),
    )
    .reset_index()
)

def summarize_numeric(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Summarize only numeric metric columns; keep string IDs out of mean/std aggregation."""
    numeric_cols = [
        col for col in df.select_dtypes(include=[np.number]).columns
        if col not in set(group_cols)
    ]
    summary = df.groupby(group_cols, dropna=False)[numeric_cols].agg(["mean", "std", "median", "count"])
    summary.columns = ["_".join(col).strip("_") for col in summary.columns]
    return summary.reset_index()

spatial_dataset_summary = summarize_numeric(
    spatial_sample_summary,
    ["model_family", "model_id", "target_layer", "region"],
)
temporal_dataset_summary = summarize_numeric(
    temporal_sample_summary,
    ["model_family", "model_id", "target_layer"],
)

spatial_sample_summary.to_csv(METRICS_DIR / "motion_saliency_spatial_sample_summary.csv", index=False)
temporal_sample_summary.to_csv(METRICS_DIR / "motion_saliency_temporal_sample_summary.csv", index=False)
spatial_dataset_summary.to_csv(METRICS_DIR / "motion_saliency_spatial_dataset_summary.csv", index=False)
temporal_dataset_summary.to_csv(METRICS_DIR / "motion_saliency_temporal_dataset_summary.csv", index=False)

# Backward-compatible combined summary file for the output checks/download scripts.
dataset_summary = spatial_dataset_summary.copy()
dataset_summary.to_csv(METRICS_DIR / "motion_saliency_dataset_summary.csv", index=False)

display(spatial_dataset_summary.head())
display(temporal_dataset_summary.head())


## Representative Visualizations

Only a small number of figures are saved. Each figure shows selected transitions with frame pair, flow magnitude, transition Grad-CAM saliency, and a combined saliency/flow overlay.

In [ ]:
def normalize01(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    x = x - float(np.nanmin(x))
    mx = float(np.nanmax(x))
    return x / mx if mx > 0 else np.zeros_like(x)


def overlay_heat(gray: np.ndarray, heat: np.ndarray, cmap_name: str = "magma", alpha: float = 0.45) -> np.ndarray:
    gray_rgb = np.repeat(np.clip(gray[..., None], 0, 1), 3, axis=-1)
    heat = normalize01(heat)
    cmap = plt.get_cmap(cmap_name)
    color = cmap(heat)[..., :3]
    local_alpha = alpha * heat[..., None]
    return np.clip(gray_rgb * (1 - local_alpha) + color * local_alpha, 0, 1)


def save_qualitative_for_row(cam_row: Any, flow_lookup: dict[str, Any]) -> list[dict[str, str]]:
    frame_indices = parse_frame_indices(cam_row.frame_indices)
    key = sequence_key(str(cam_row.video_id), frame_indices)
    flow_path = Path(flow_lookup[key]["flow_npz_path"])
    cam_path = Path(cam_row.resolved_cam_npz_path)
    cams = load_cam_array(cam_path, str(CONFIG["analysis_cam_key"]))
    sal = transition_saliency(cams)
    with np.load(flow_path, allow_pickle=False) as flow:
        frames = flow["frames"].astype(np.float32)
        mag = flow["flow_forward_magnitude"].astype(np.float32)
        cavity = flow["transition_cavity_roi"].astype(bool)

    transitions = sal.shape[0]
    selected = np.linspace(0, transitions - 1, min(int(CONFIG["visualization_transition_count"]), transitions)).round().astype(int)
    fig, axes = plt.subplots(len(selected), 4, figsize=(12, 2.8 * len(selected)), squeeze=False)
    for row_idx, t in enumerate(selected):
        axes[row_idx, 0].imshow(frames[t], cmap="gray", vmin=0, vmax=1)
        axes[row_idx, 0].set_title(f"frame {frame_indices[t]}", fontsize=8)
        axes[row_idx, 1].imshow(overlay_heat(frames[t], mag[t], cmap_name="viridis", alpha=0.55))
        axes[row_idx, 1].set_title(f"flow {frame_indices[t]}->{frame_indices[t+1]}", fontsize=8)
        axes[row_idx, 2].imshow(overlay_heat(frames[t], sal[t], cmap_name="magma", alpha=0.55))
        axes[row_idx, 2].set_title("transition CAM", fontsize=8)
        combo = overlay_heat(overlay_heat(frames[t], mag[t], cmap_name="viridis", alpha=0.35).mean(axis=-1), sal[t], cmap_name="magma", alpha=0.55)
        axes[row_idx, 3].imshow(combo)
        if cavity[t].any():
            axes[row_idx, 3].contour(cavity[t], levels=[0.5], colors="cyan", linewidths=0.8)
        axes[row_idx, 3].set_title("CAM + flow + LV ROI", fontsize=8)
        for ax in axes[row_idx]:
            ax.axis("off")
    fig.suptitle(f"{cam_row.sample_id} | {cam_row.model_id} | {cam_row.target_layer}", fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    out_dir = QUAL_DIR / str(cam_row.target_layer) / str(cam_row.model_id)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{cam_row.sample_id}_{cam_row.model_id}_{cam_row.target_layer}_flow_saliency.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return [{"sample_id": str(cam_row.sample_id), "model_id": str(cam_row.model_id), "target_layer": str(cam_row.target_layer), "figure_path": str(out_path)}]

qual_rows = []
for sample_id in tqdm(visualization_sample_ids, desc="qualitative examples"):
    sample_rows = cam_manifest[cam_manifest["sample_id"].astype(str) == str(sample_id)].copy()
    if sample_rows.empty:
        continue
    # Prefer ConvLSTM layers, but keep at most a manageable number per sample.
    sample_rows["layer_priority"] = sample_rows["target_layer"].apply(lambda x: CONFIG["prefer_layers"].index(x) if x in CONFIG["prefer_layers"] else 999)
    sample_rows = sample_rows.sort_values(["layer_priority", "model_id", "target_layer"]).head(4)
    for cam_row in sample_rows.itertuples(index=False):
        qual_rows.extend(save_qualitative_for_row(cam_row, flow_lookup))

qualitative_df = pd.DataFrame(qual_rows)
qualitative_df.to_csv(MANIFEST_DIR / "qualitative_examples_manifest.csv", index=False)
display(qualitative_df.head())

## Output Checks

In [ ]:
required = [
    MANIFEST_DIR / "flow_manifest.csv",
    METRICS_DIR / "flow_transition_metrics.csv",
    METRICS_DIR / "motion_saliency_spatial_transition_metrics.csv",
    METRICS_DIR / "motion_saliency_temporal_transition_metrics.csv",
    METRICS_DIR / "motion_saliency_spatial_sample_summary.csv",
    METRICS_DIR / "motion_saliency_temporal_sample_summary.csv",
    METRICS_DIR / "motion_saliency_dataset_summary.csv",
    MANIFEST_DIR / "qualitative_examples_manifest.csv",
]
missing = [path for path in required if not path.exists()]
assert not missing, f"Missing expected outputs: {missing}"
summary = {
    "run_mode": RUN_MODE,
    "gradcam_run_dir": str(GRADCAM_RUN_DIR),
    "flow_method": CONFIG["flow_method"],
    "cam_rows": int(len(cam_manifest)),
    "unique_flow_sequences": int(len(sequence_df)),
    "spatial_metric_rows": int(len(spatial_metrics_df)),
    "temporal_metric_rows": int(len(temporal_metrics_df)),
    "note": "Grad-CAM was not regenerated; saved notebook 05 CAM arrays were loaded. Optical flow was computed once per unique frame sequence and reused for every model/layer.",
}
with (OUTPUT_DIR / "faithfulness_summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))